# Ward views

Ward-level maps built from the tables in `data/tables/` (`python scripts/build_tables.py all`) and the
ward boundaries in `data/raw/layers/` (`scripts/fetch_layers.py`). Each section shows the table it draws
next to the plot. Drawing helpers shared by every section: `scripts/ward_maps.py`.
**This notebook so far: section 1 only.**

| § | View | Reads |
|---|---|---|
| 1 | Alder map (ward number + last name) and photo grid | `people`, `data/raw/photos/` (`scripts/fetch_photos.py`) |
| 2 | Tenure map + sorted bar chart | *not built yet* |
| 3 | Absence map + ranked table | *not built yet* |
| 4 | Split roll calls: table + one map colored by vote | *not built yet* |

Kernel: **Chicago Council**.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, "../scripts")
import ward_maps

# --- Settings (every choice made in this notebook) --------------------------

TABLES_DIR = Path("../data/tables")
PHOTOS_DIR = Path("../data/raw/photos")

# Ward shapes: "real" (2023 boundaries) or "tiles" (plan Phase 4, not built yet).
MAP_SHAPES = "real"

# Basemap under every map: "street", "gray" or "none" (ward_maps.BASEMAP_OPTIONS).
# User, 2026-10-04: gray. §1.3 draws all three side by side.
BASEMAP_CHOICE = "gray"

# Name shown under the ward number (user, 2026-10-04: last name only, keep "Jr.").
#   "last"      = text before the LAST comma of display_name, i.e. everything but the given name
#                 ("Martin, Matthew J." -> "Martin"; "Cardona, Jr., Felix" -> "Cardona, Jr.")
#   "as_stored" = display_name unchanged
ALDER_LABEL_NAME = "last"

FIGURE_SIZE_MAP = (12, 15)

# Label font on the §1.2 map (user, 2026-10-04: as large as possible). 9 is the largest size for
# which hand nudges were found that leave no two labels overlapping; §1.2's sweep shows counts by size.
MAP_LABEL_FONT_SIZE = 9

# Hand nudges for labels that would otherwise overlap at MAP_LABEL_FONT_SIZE on FIGURE_SIZE_MAP.
# {ward: (right, up)} in typographic points; negative = left / down. Wards not listed sit at
# their representative point. Set by eye on 2026-10-04, checked by ward_maps.label_overlaps.
LABEL_OFFSET_POINTS_BY_WARD = {
    "40": (-4, -22),   # Vasquez, Jr.: down into the lower part of the narrow ward, clear of 48 and 50
    "48": (28, 8),     # Manaa-Hoppenworth: right and up toward the lakefront (text extends over the lake)
    "33": (-12, 0),    # Rodriguez Sanchez: left, clear of 47
    "47": (8, 0),      # Martin: right, clear of 33
    "14": (0, 4),      # Gutierrez: up, clear of 15
    "15": (6, -12),    # Lopez: down and right, clear of 14
    "03": (0, -5),     # Dowell: down, clear of 4
    "04": (0, 5),      # Robinson: up, clear of 3
    "43": (10, 3),     # Knudsen: right and slightly up, clear of 32 and 2
    "35": (-5, 0),     # Quezada: left, clear of 32
    "50": (0, -3),     # Silverstein: down, clear of 49
    "49": (0, 2),      # Hadden: up, clear of 50
}

# Font sizes tried in §1.2's sweep (overlap count with and without the nudges above).
LABEL_FONT_SIZES_TO_SWEEP = [6, 7, 8, 9, 10, 11, 12]

# Photo grid display (files on disk are never changed).
#   PHOTO_DISPLAY_CROP: "square" = crop non-square photos to a square (user, 2026-10-04); "none" = as is
#   PHOTO_CROP_VERTICAL_ANCHOR: for a tall photo, "top" keeps the top (head room like the square
#   photos), "center" cuts top and bottom equally. §1.4 shows both for every non-square photo.
PHOTO_DISPLAY_CROP = "square"
PHOTO_CROP_VERTICAL_ANCHOR = "top"
FIGURE_SIZE_BASEMAP_COMPARISON = (24, 11)
PHOTO_GRID_COLUMNS = 10

pd.set_option("display.max_rows", 200)

## 1. Alder map

### 1.1 Current alder per ward — the table the map draws

Current alder = `people` row with `is_active` = True and a ward number in `elms_ward` (decision 1.1:
eLMS `/person`). The same rule picks whose photo `scripts/fetch_photos.py` downloads. Clerk and Mayor
rows are active but have no ward and are left off the map; they are listed below so nothing is
hidden.

In [ ]:
people_table = pd.read_csv(TABLES_DIR / "people.csv", dtype={"elms_ward": str})
has_ward_number = people_table.elms_ward.str.fullmatch(r"\d+", na=False)
current_alders = people_table[people_table.is_active & has_ward_number].copy()

print("active people without a ward number (not on the map):")
display(people_table[people_table.is_active & ~has_ward_number][["person_id", "display_name", "elms_ward"]])

# Label name, per ALDER_LABEL_NAME.
if ALDER_LABEL_NAME == "last":
    current_alders["label_name"] = current_alders.display_name.str.rsplit(",", n=1).str[0].str.strip()
elif ALDER_LABEL_NAME == "as_stored":
    current_alders["label_name"] = current_alders.display_name
else:
    raise ValueError(ALDER_LABEL_NAME)
current_alders["comma_count"] = current_alders.display_name.str.count(",")

print(f"current alders: {len(current_alders)}; distinct wards: {current_alders.elms_ward.nunique()}")
print("names whose display_name does not have exactly one comma (check their label):")
display(current_alders[current_alders.comma_count != 1][["elms_ward", "display_name", "label_name"]])

In [ ]:
# Join to ward shapes. Ward key: people.elms_ward "01" <-> layer ward "1", padded in ward_maps.load_ward_shapes.
ward_shapes = ward_maps.load_ward_shapes(MAP_SHAPES)
alder_map_table = ward_shapes.merge(current_alders.rename(columns={"elms_ward": "ward"}),
                                    on="ward", how="outer", indicator=True)
print("join result (both = ward shape and alder matched):", alder_map_table._merge.value_counts().to_dict())
assert (alder_map_table._merge == "both").all() and len(alder_map_table) == 50

alder_map_table = alder_map_table.sort_values("ward").reset_index(drop=True)
alder_map_table["map_label"] = alder_map_table.ward.str.lstrip("0") + "\n" + alder_map_table.label_name
display(alder_map_table[["ward", "ward_raw", "person_id", "display_name", "label_name"]])

### 1.2 Map — ward number and alder name

Labels sit at each ward's `representative_point()` (a point guaranteed inside the shape, which is
not always the visual center), moved by `LABEL_OFFSET_POINTS_BY_WARD` where listed. Font:
`MAP_LABEL_FONT_SIZE`. Basemap: `BASEMAP_CHOICE`. Some long names extend past their ward's outline.
After the map: the list of overlapping label pairs (expected: none), then a font-size sweep.

In [ ]:
map_label_by_ward = dict(zip(alder_map_table.ward, alder_map_table.map_label))

figure, axis = plt.subplots(figsize=FIGURE_SIZE_MAP)
ward_maps.draw_ward_outlines(axis, alder_map_table)
annotation_by_ward = ward_maps.draw_ward_labels(axis, alder_map_table, map_label_by_ward, font_size=MAP_LABEL_FONT_SIZE,
                                                offset_points_by_ward=LABEL_OFFSET_POINTS_BY_WARD)
ward_maps.add_basemap(axis, BASEMAP_CHOICE)
axis.set_title(f"Chicago City Council: ward and current alder (people.csv, eLMS /person) — basemap: {BASEMAP_CHOICE}")
axis.set_axis_off()
figure.canvas.draw()
map_label_overlaps = ward_maps.label_overlaps(figure, annotation_by_ward)
plt.show()

print(f"overlapping label pairs on this map: {len(map_label_overlaps)}")
display(pd.DataFrame(map_label_overlaps, columns=["ward_a", "ward_b", "overlap_width_px", "overlap_height_px"]))

In [ ]:
# Font-size sweep: overlapping label pairs at each size, without and with LABEL_OFFSET_POINTS_BY_WARD
# (same figure size, no basemap; the basemap does not move labels). The nudges were set for
# MAP_LABEL_FONT_SIZE, so at other sizes they are not expected to be enough.
sweep_rows = []
for font_size in LABEL_FONT_SIZES_TO_SWEEP:
    for nudges_name, nudges in [("no nudges", {}), ("with nudges", LABEL_OFFSET_POINTS_BY_WARD)]:
        sweep_figure, sweep_axis = plt.subplots(figsize=FIGURE_SIZE_MAP)
        ward_maps.draw_ward_outlines(sweep_axis, alder_map_table)
        sweep_annotations = ward_maps.draw_ward_labels(sweep_axis, alder_map_table, map_label_by_ward,
                                                       font_size=font_size, offset_points_by_ward=nudges)
        sweep_axis.set_axis_off()
        sweep_figure.canvas.draw()
        overlaps = ward_maps.label_overlaps(sweep_figure, sweep_annotations)
        plt.close(sweep_figure)
        sweep_rows.append({"font_size": font_size, "nudges": nudges_name, "overlapping_pairs": len(overlaps),
                           "pairs": ", ".join(f"{ward_a}/{ward_b}" for ward_a, ward_b, *_ in overlaps)})
display(pd.DataFrame(sweep_rows))

### 1.3 Basemap options side by side

The same map with each entry of `ward_maps.BASEMAP_OPTIONS`. Pick one for `BASEMAP_CHOICE`
in the settings cell. Panels are smaller than §1.2, so labels are 5 pt with no nudges.

In [ ]:
figure, axes = plt.subplots(1, len(ward_maps.BASEMAP_OPTIONS), figsize=FIGURE_SIZE_BASEMAP_COMPARISON)
for axis, basemap_option in zip(axes, ward_maps.BASEMAP_OPTIONS):
    ward_maps.draw_ward_outlines(axis, alder_map_table)
    ward_maps.draw_ward_labels(axis, alder_map_table, dict(zip(alder_map_table.ward, alder_map_table.map_label)),
                               font_size=5)
    ward_maps.add_basemap(axis, basemap_option)
    axis.set_title(f'BASEMAP_CHOICE = "{basemap_option}"')
    axis.set_axis_off()
plt.tight_layout()
plt.show()

### 1.4 Photo grid

Photos from `data/raw/photos/` (`scripts/fetch_photos.py`, from `people.photo_url`, saved unchanged).
The manifest table shows each file's source, retrieval time, size and pixel dimensions. The files
differ a lot in resolution. The grid draws each photo at the same cell size; non-square photos are
cropped for display per `PHOTO_DISPLAY_CROP` / `PHOTO_CROP_VERTICAL_ANCHOR` (files unchanged).

In [ ]:
photo_manifest = pd.DataFrame(json.loads((PHOTOS_DIR / "manifest.json").read_text())).T
photo_manifest.index.name = "person_id"
photo_manifest["pixel_width_height"] = [Image.open(PHOTOS_DIR / file_name).size for file_name in photo_manifest.file]

# Every current alder has exactly one photo, and no extra photos.
assert set(photo_manifest.index) == set(alder_map_table.person_id), "photo set differs from current alders"
photo_manifest = photo_manifest.sort_values("elms_ward")
display(photo_manifest[["elms_ward", "display_name", "file", "content_type", "bytes", "pixel_width_height", "retrieved_at"]])

In [ ]:
# Every non-square photo: as stored, and both crop anchors, so PHOTO_CROP_VERTICAL_ANCHOR can be checked.
non_square_photos = photo_manifest[[width != height for width, height in photo_manifest.pixel_width_height]]
print(f"non-square photos: {len(non_square_photos)}")
crop_versions = ["as stored", "top", "center"]
figure, axes_grid = plt.subplots(len(non_square_photos), len(crop_versions), figsize=(7, 3 * len(non_square_photos)),
                                 squeeze=False)
for axes_row, photo in zip(axes_grid, non_square_photos.itertuples()):
    image = Image.open(PHOTOS_DIR / photo.file)
    for axis, crop_version in zip(axes_row, crop_versions):
        shown = image if crop_version == "as stored" else ward_maps.crop_to_square(image, crop_version)
        axis.imshow(shown)
        axis.set_title(f"{photo.display_name}\n{crop_version} {shown.size}", fontsize=8)
        axis.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
def photo_for_display(person_id):
    image = Image.open(PHOTOS_DIR / photo_manifest.loc[person_id, "file"])
    if PHOTO_DISPLAY_CROP == "square":
        return ward_maps.crop_to_square(image, PHOTO_CROP_VERTICAL_ANCHOR)
    if PHOTO_DISPLAY_CROP == "none":
        return image
    raise ValueError(PHOTO_DISPLAY_CROP)


row_count = -(-len(alder_map_table) // PHOTO_GRID_COLUMNS)   # ceiling division
figure, axes_grid = plt.subplots(row_count, PHOTO_GRID_COLUMNS, figsize=(2 * PHOTO_GRID_COLUMNS, 2.6 * row_count))
for axis, alder in zip(axes_grid.flat, alder_map_table.itertuples()):
    axis.imshow(photo_for_display(alder.person_id))
    axis.set_title(f"{alder.ward.lstrip('0')} — {alder.label_name}", fontsize=9)
    axis.set_axis_off()
for unused_axis in list(axes_grid.flat)[len(alder_map_table):]:
    unused_axis.set_axis_off()
plt.tight_layout()
plt.show()